In [5]:
import numpy as np

import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV


In [6]:
# Load features
X_train_hog = np.load("../features/X_train_hog_scaled.npy")
X_val_hog   = np.load("../features/X_val_hog_scaled.npy")
X_test_hog  = np.load("../features/X_test_hog_scaled.npy")

X_train_cnn = np.load("../features/X_train_cnn_scaled.npy")
X_val_cnn   = np.load("../features/X_val_cnn_scaled.npy")
X_test_cnn  = np.load("../features/X_test_cnn_scaled.npy")

X_train_hc = np.load("../features/X_train_scaled.npy")
X_val_hc   = np.load("../features/X_val_scaled.npy")
X_test_hc  = np.load("../features/X_test_scaled.npy")

y_train = np.load("../features/y_train.npy", allow_pickle=True)
y_val   = np.load("../features/y_val.npy", allow_pickle=True)
y_test  = np.load("../features/y_test.npy", allow_pickle=True)

In [7]:
param_grid = {
    "n_neighbors": [7],
    "weights": ["distance"],
    "metric": ["euclidean"]
}

def train_knn(X_train, y_train):
    grid = GridSearchCV(
        KNeighborsClassifier(),
        param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    return grid.best_estimator_, grid.best_score_

In [8]:
def train_and_eval(X_train, X_val, X_test, y_train, y_val, y_test, label=""):
    model, best_cv_score = train_knn(X_train, y_train)

    val_acc  = accuracy_score(y_val,  model.predict(X_val))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    print(f"\n{label}")
    print("Validation Accuracy:", val_acc)
    print("Test Accuracy:", test_acc)

    return model, val_acc, test_acc

In [9]:

hog_model, hog_val, hog_test = train_and_eval(
    X_train_hog, X_val_hog, X_test_hog,
    y_train, y_val, y_test,
    "HOG ONLY"
)

cnn_model, cnn_val, cnn_test = train_and_eval(
    X_train_cnn, X_val_cnn, X_test_cnn,
    y_train, y_val, y_test,
    "CNN ONLY"
)

fusion_model, fusion_val, fusion_test = train_and_eval(
    X_train_hc, X_val_hc, X_test_hc,
    y_train, y_val, y_test,
    "HOG + CNN"
)


HOG ONLY
Validation Accuracy: 0.5806451612903226
Test Accuracy: 0.6148409893992933

CNN ONLY
Validation Accuracy: 0.8673835125448028
Test Accuracy: 0.8833922261484098

HOG + CNN
Validation Accuracy: 0.8853046594982079
Test Accuracy: 0.8869257950530035


In [10]:
results = {
    "HOG": (hog_model, hog_val, hog_test),
    "CNN": (cnn_model, cnn_val, cnn_test),
    "HOG+CNN": (fusion_model, fusion_val, fusion_test)
}

# choose best model using validation accuracy
best_name = max(results, key=lambda x: results[x][1])

best_model = results[best_name][0]
best_val   = results[best_name][1]
best_test  = results[best_name][2]

print("\n========== BEST MODEL ==========")
print("Best Feature Type :", best_name)
print("Validation Accuracy :", round(best_val, 4))
print("Test Accuracy       :", round(best_test, 4))


========== BEST MODEL ==========
Best Feature Type : HOG+CNN
Validation Accuracy : 0.8853
Test Accuracy       : 0.8869


In [11]:

CLASS_MAPPING = {
    "cardboard": 0,
    "glass": 1,
    "metal": 2,
    "paper": 3,
    "plastic": 4,
    "trash": 5,
    "unknown": 6
}

UNKNOWN_LABEL = CLASS_MAPPING["unknown"]

X_train = np.load("../features/X_train_scaled.npy")
X_val   = np.load("../features/X_val_scaled.npy")
X_test  = np.load("../features/X_test_scaled.npy")

y_train_str = np.load("../features/y_train.npy", allow_pickle=True)
y_val_str   = np.load("../features/y_val.npy", allow_pickle=True)
y_test_str  = np.load("../features/y_test.npy", allow_pickle=True)

y_train = np.array([CLASS_MAPPING[y] for y in y_train_str])
y_val   = np.array([CLASS_MAPPING[y] for y in y_val_str])
y_test  = np.array([CLASS_MAPPING[y] for y in y_test_str])



In [12]:
knn = KNeighborsClassifier(
    n_neighbors=7,
    weights="distance",
    metric="euclidean"
)

knn.fit(X_train, y_train)

y_val_pred = knn.predict(X_val)
y_test_pred = knn.predict(X_test)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))



Validation Accuracy: 0.8853046594982079
Test Accuracy: 0.8869257950530035


In [13]:
def knn_predict_with_rejection(model, X, threshold=0.6):
    distances, indices = model.kneighbors(X)
    neighbor_labels = model._y[indices]

    final_preds = []

    for i in range(len(X)):
        labels, counts = np.unique(neighbor_labels[i], return_counts=True)

        best_idx = np.argmax(counts)
        best_label = labels[best_idx]
        confidence = counts[best_idx] / model.n_neighbors

        if confidence < threshold:
            final_preds.append(UNKNOWN_LABEL)
        else:
            final_preds.append(best_label)

    return np.array(final_preds)



In [14]:
y_test_rej = knn_predict_with_rejection(knn, X_test, threshold=0.5)

mask = y_test_rej != UNKNOWN_LABEL

if np.sum(mask) == 0:
    print("All samples rejected → decrease threshold")
else:
    acc_rej = accuracy_score(y_test[mask], y_test_rej[mask])
    print("Accuracy with rejection:", acc_rej)
    print("Rejected samples:", np.sum(~mask))

    print("\nClassification Report (No Rejection):")
print(classification_report(y_test, y_test_pred))

Accuracy with rejection: 0.9007352941176471
Rejected samples: 11

Classification Report (No Rejection):
              precision    recall  f1-score   support

           0       0.97      0.95      0.96        38
           1       0.85      0.86      0.85        58
           2       0.86      0.90      0.88        48
           3       0.94      0.96      0.95        68
           4       0.87      0.85      0.86        55
           5       0.71      0.62      0.67        16

    accuracy                           0.89       283
   macro avg       0.87      0.86      0.86       283
weighted avg       0.89      0.89      0.89       283



In [15]:
joblib.dump(best_model, "../models/best_knn_model.pkl")

print("\nModel saved successfully!")


Model saved successfully!
